# Notebook 3 — Non-CMF Unlearning Baselines

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Methods:** NegGrad+, Random-label, SalUn, SCRUB, UNSIR, SVD  
**Prerequisites:** Notebook 1 output attached. Set `CKPT_DATASET_DIR` below.

**Hyperparameters:** All from paper Tables 4/5, sourced via `paper_hparams.py`.  
**Split protocol:** Loads the SAME files generated once in NB1 — never re-splits (A.6 fix).  
**forget_class identification:** derived from split index sets, NOT from `unlearn_class=[]` (A.3 fix).

**Outputs per (method, ratio, seed):**  
`{method}_nocmf_ratio{30|10}_seed{seed}.pt` with embedded Output/Probe/NCC metrics.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/ycgao1/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git','-C',REPO_DIR,'rev-parse','HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
# ══ SET THIS to the Kaggle dataset mount path from Notebook 1 ══
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-notebook1'

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, f'Cannot find cmf_benchmark_config.json under {CKPT_DATASET_DIR}'
with open(config_path) as f: NB1_CFG = json.load(f)

DATASET     = NB1_CFG['dataset']
ARCH        = NB1_CFG['arch']
NUM_CLASSES = NB1_CFG['num_classes']
SEEDS       = NB1_CFG['seeds']
RATIOS      = NB1_CFG['ratios']
TEST_MODE   = NB1_CFG.get('test_mode', False)

CKPT_ROOT = '/kaggle/working/checkpoints/unlearn_nocmf'
os.makedirs(CKPT_ROOT, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DATASET={DATASET}  ARCH={ARCH}  NUM_CLASSES={NUM_CLASSES}  device={device}')

In [ ]:
import torchvision, torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
full_train = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                          download=True, transform=transform_train)
# full_train_eval: same images as full_train but with test-time transform (no aug).
# Used to evaluate forget/retain accuracy on training-set indices.
full_train_eval = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                               download=False, transform=transform_test)
test_set   = torchvision.datasets.CIFAR10('/kaggle/working/data', train=False,
                                          download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
full_train_loader = torch.utils.data.DataLoader(full_train, batch_size=256, shuffle=False, num_workers=2)
print(f'Train: {len(full_train)}  Test: {len(test_set)}')

In [ ]:
from models.resnet import ResNet18

def build_model():
    return ResNet18(num_classes=NUM_CLASSES, dataset=DATASET).to(device)

@torch.no_grad()
def extract_features_resnet(model, loader):
    """Extract avgpool features via forward hook — works for both
    ResNet_cifar (no maxpool) and standard ResNet (has maxpool)."""
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        buf = []
        hook = model.avgpool.register_forward_hook(
            lambda m, i, o: buf.append(o.flatten(1).detach().cpu())
        )
        model(x)
        hook.remove()
        feats.append(buf[0]); labs.append(y)
    return torch.cat(feats), torch.cat(labs)

@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)

def run_linear_probe(model, retain_ldr, forget_ldr, n_epochs=50, lr=1e-2):
    """Train a fresh linear head on D_r ∪ D_f features (paper §3.2),
    then evaluate separately on retain and forget splits."""
    Xtr, ytr = extract_features_resnet(model, retain_ldr)
    Xfg, yfg = extract_features_resnet(model, forget_ldr)
    # Paper §3.2: probe trains on full D = D_r ∪ D_f
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=lr, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        ret_acc = (head(Xtr.to(device)).argmax(1).cpu() == ytr).float().mean().item() * 100
        fgt_acc = (head(Xfg.to(device)).argmax(1).cpu() == yfg).float().mean().item() * 100
    return ret_acc, fgt_acc

def run_ncc(model, retain_ldr, forget_ldr):
    """NCC (NC-4): nearest-class-centre by Euclidean distance in raw feature space.
    Matches evaluation/nc.py::ncc_accuracy_from_features() exactly.
    Class means μ_k built from D_r ∪ D_f (paper eq.3: all training samples).
    Evaluated separately on retain and forget.
    """
    Xtr, ytr = extract_features_resnet(model, retain_ldr)   # raw avgpool features
    Xfg, yfg = extract_features_resnet(model, forget_ldr)
    # Paper eq.3: μ_k = mean over ALL training samples (D_r ∪ D_f)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)   # [C, D]
    # Euclidean distance → argmin  (canonical NCC)
    ret_pred = torch.cdist(Xtr.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    fgt_pred = torch.cdist(Xfg.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    ret_acc = (ret_pred == ytr).float().mean().item() * 100
    fgt_acc = (fgt_pred == yfg).float().mean().item() * 100
    return ret_acc, fgt_acc

def eval_three_metrics(model, retain_eval_ldr, forget_eval_ldr, full_retain_ldr):
    """Three-metric evaluation for stratified random-mix splits.
    - Output: direct model accuracy on retain_eval / forget_eval loaders.
    - Probe:  linear head trained on D_r ∪ D_f features (paper §3.2).
    - NCC:    nearest-class-centre with means from D_r ∪ D_f (paper eq.3).
    full_retain_ldr provides retain features; forget_eval_ldr provides forget features.
    """
    out_ret = eval_acc(model, retain_eval_ldr)
    out_fgt = eval_acc(model, forget_eval_ldr)
    lp_ret, lp_fgt   = run_linear_probe(model, full_retain_ldr, forget_eval_ldr)
    ncc_ret, ncc_fgt = run_ncc(model, full_retain_ldr, forget_eval_ldr)
    return {'output_retain_acc':out_ret,'output_forget_acc':out_fgt,
            'probe_retain_acc':lp_ret,  'probe_forget_acc':lp_fgt,
            'ncc_retain_acc':ncc_ret,   'ncc_forget_acc':ncc_fgt}

print('Eval helpers ready.')

In [ ]:
# A.3 fix: ALL values from paper Table 4 exclusively (no shell-script values)
# Source: Table 4, PDF lines 2330-2388  (arXiv:2604.08271v1)
# hparam_source tag lets us distinguish these from earlier shell-derived checkpoints
HPARAM_SOURCE = 'table4'
METHODS = [
    # (method_key, dispatch_key, epochs, lr_cifar10, batch_size)
    ('random_label',          'random_label',          3,  1e-4,  128),  # Table 4 line 2330: lr=1e-4
    ('salun',                 'salun',                 3,  1e-4,  128),  # Table 4 line 2339: lr=1e-4
    ('grad_ascent_descent',   'grad_ascent_descent',   3,  1e-4,  128),  # Table 4 line 2348: lr=1e-4; NegGrad+
    ('scrub',                 'scrub',                 3,  1e-4,   64),  # Table 4 line 2357: lr=1e-4; batch=64
    ('tarun',                 'tarun',                 3,  5e-5,  128),  # Table 4 line 2366: lr=5e-5; UNSIR full-model
    ('SVD',                   'SVD',                   1,  0.0,   900),  # Table 4 line 2375: training-free
]

def make_unlearn_args(method_key, lr, epochs, batch_size, forget_idx, retain_idx, seed=0):
    """Build args namespace for unlearning. All values from paper Table 4."""
    import argparse
    # A.3 fix: forget classes derived from forget_idx (stratified: spans all 10 classes).
    # utils.test() ZeroDivisionError when all classes are in unlearn_class is fixed
    # directly in utils.py (safe retain denominator guard). So we pass real classes here.
    forget_classes = list(set(full_train.targets[i] for i in forget_idx))
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=method_key,
        unlearn_class=forget_classes,   # real forget classes
        batch_size=batch_size, test_batch_size=256,
        lr=lr, momentum=0.9, weight_decay=5e-4,
        epochs_or_steps=epochs,
        seed=seed,
        # sample counts
        num_retain_samples=len(retain_idx),
        num_forget_samples=len(forget_idx),
        # gradient clipping (Table 4: NegGrad+ uses grad-clip=1.0)
        grad_norm_clip=(1.0 if 'grad_ascent' in method_key else None),
        # SVD params (Table 4 line 2375: CIFAR-10 alpha_r=1000, alpha_f=30, samples=900)
        SVD_alpha_r=1000, SVD_alpha_f=30, SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,
        # SCRUB params (Table 4 line 2357: sgda-bsz=64, msteps=2)
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        # SalUn params (Table 4 line 2339: threshold=0.5)
        salun_threshold=0.5,
        # UNSIR params (Table 4 line 2366: full-model; impair_lr = same as main lr = 5e-5)
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        # misc
        dry_run=False, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data',
        remove_FC=False, CMFClassifier=False,
        prob_batch_size=256, lp_every=0, ncc_every=0,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

print('Method helpers ready.')

In [ ]:
from unlearn import unlear_func

all_results = []

for ratio in RATIOS:
    for seed in SEEDS:
        tag_split = f'ratio{ratio}_seed{seed}'
        if TEST_MODE: tag_split += '_testmode'

        # A.6 fix: load split from NB1, never re-generate
        fpath = f'{CKPT_ROOT_NB1}/splits/forget_indices_{tag_split}.json'
        rpath = f'{CKPT_ROOT_NB1}/splits/retain_indices_{tag_split}.json'
        assert os.path.exists(fpath), f'Missing split: {fpath}'
        with open(fpath) as f: forget_idx = json.load(f)
        with open(rpath) as f: retain_idx = json.load(f)

        # Load theta_o checkpoint
        theta_o_path = f'{CKPT_ROOT_NB1}/pre_train/theta_o_seed{seed}.pt'
        if TEST_MODE: theta_o_path = theta_o_path.replace('.pt', '_testmode.pt')
        assert os.path.exists(theta_o_path), f'Missing theta_o: {theta_o_path}'

        # Stratified split: forget_idx/retain_idx are training indices.
        retain_eval_set = torch.utils.data.Subset(full_train_eval, retain_idx)
        forget_eval_set = torch.utils.data.Subset(full_train_eval, forget_idx)
        retain_eval_ldr = torch.utils.data.DataLoader(retain_eval_set, batch_size=256, shuffle=False, num_workers=2)
        forget_eval_ldr = torch.utils.data.DataLoader(forget_eval_set, batch_size=256, shuffle=False, num_workers=2)

        # Augmented retain loader (for unlearning training + probe/NCC feature extraction)
        retain_set  = torch.utils.data.Subset(full_train, retain_idx)
        forget_set  = torch.utils.data.Subset(full_train, forget_idx)
        retain_loader = torch.utils.data.DataLoader(retain_set, batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(forget_set, batch_size=128, shuffle=True, num_workers=2)

        for method_key, dispatch_key, epochs, lr, batch_sz in METHODS:
            tag = f'{method_key}_nocmf_ratio{ratio}_seed{seed}'
            if TEST_MODE: tag += '_testmode'
            ckpt_path = f'{CKPT_ROOT}/{tag}.pt'

            # Resumable: skip if checkpoint already exists
            if os.path.exists(ckpt_path):
                print(f'[{tag}] exists — skipping.')
                ck = torch.load(ckpt_path, map_location=device)
                all_results.append(ck['metrics'])
                continue

            print(f'\n[{tag}] Running {method_key} lr={lr} epochs={epochs} batch={batch_sz}')
            torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

            # Load fresh theta_o
            model = build_model()
            ck_o  = torch.load(theta_o_path, map_location=device)
            model.load_state_dict(ck_o['model_state_dict'] if 'model_state_dict' in ck_o else ck_o)

            args = make_unlearn_args(method_key, lr, epochs, batch_sz, forget_idx, retain_idx, seed=seed)
            fn   = unlear_func[dispatch_key]

            t0 = time.time()
            try:
                model = fn(
                    args=args, model=model, device=device,
                    retain_loader=retain_loader, forget_loader=forget_loader,
                    train_loader=full_train_loader, test_loader=test_loader,
                    optimizer=None, epochs=epochs,
                    test_forget_loader=forget_loader,
                    train_dataset=full_train,   # required by tarun_unlearn, SVD_unlearn
                    val_index=retain_idx,       # required by tarun_unlearn, SVD_unlearn
                )
            except Exception as e:
                print(f'  ERROR: {e}')
                continue
            wall_min = (time.time() - t0) / 60

            metrics = eval_three_metrics(model, retain_eval_ldr, forget_eval_ldr, retain_loader)
            metrics.update({'method': method_key, 'ratio': ratio, 'seed': seed,
                            'lr': lr, 'epochs': epochs, 'batch_size': batch_sz,
                            'wall_clock_minutes': wall_min,
                            'hparam_source': HPARAM_SOURCE})

            torch.save({
                'model_state_dict': model.state_dict(),
                'config': {
                    'method': method_key, 'dataset': DATASET, 'arch': ARCH,
                    'ratio': ratio, 'seed': seed, 'lr': lr, 'epochs': epochs,
                    'num_classes': NUM_CLASSES,
                    'split_source': 'nb1', 'mean_source': 'N/A',
                    'hparam_source': HPARAM_SOURCE,
                    'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                },
                'seed': seed,
                'metrics': metrics,
            }, ckpt_path)
            print(f'  Saved {ckpt_path}')
            print(f'  output R={metrics["output_retain_acc"]:.2f}% F={metrics["output_forget_acc"]:.2f}%  '
                  f'probe R={metrics["probe_retain_acc"]:.2f}% F={metrics["probe_forget_acc"]:.2f}%  '
                  f'ncc R={metrics["ncc_retain_acc"]:.2f}% F={metrics["ncc_forget_acc"]:.2f}%')
            all_results.append(metrics)

df = pd.DataFrame(all_results)
csv_path = f'{CKPT_ROOT}/results_nocmf.csv'
df.to_csv(csv_path, index=False)
print(f'\nAll done. Results saved: {csv_path}')
print(df[['method','ratio','seed','output_retain_acc','output_forget_acc',
          'probe_retain_acc','probe_forget_acc',
          'ncc_retain_acc','ncc_forget_acc']].to_string(index=False))